## Python Project Part 1 - Crime

### Ingestion Layer

#### Primary Dataset

There is actually quite alot of data to consider. Thankfully, it seems to be formatted the same all throughout which makes our job easier. 

It would theoretically be possible, but certainly not smart to ingest this data in one at a time. Instead I will do so with some functions that will work together to complete the following goals:

1) Ensure that all of the data we expect to be there actually exists.

2) Check an individual CSV file.

3) Read and process each file one at a time, this will cap memory usage
because we never hold more than one month's data at once. Having worked with very large files containing scraped reddit data in the past, this step is crucial for scalability.

4) While processing files we should be keeping track of stats such    as dataset dimensions and how many files we have processed to be able to check wehther something has gone wrong.

5) Finally once we are sure that everything has been ingested correctly, we can combine all of the individual results into one final completed dataset. 


In [70]:
# importing libraries

from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd

In [71]:
#### CHECKING THAT ALL FOLDERS ARE THERE ## 

from pathlib import Path 

data_directory = Path('Full June 2023 - May 2026 Police datasets')
police_forces = ['metropolitan', 'west-midlands', 'northumbria', 'surrey']

month_folders = sorted(data_directory.glob('20*-*'))
print(f'Found {len(month_folders)} month folders')


Found 36 month folders


So far so good, this is exactly what we want, we have precisely 36 months worth of data and Python seems to have found all of it. 

The next step is to check that each month contains the four forces that we need it to. This may seem like unneccesary preproccessing because for 36 months we could theoretically check manually and be done with it.

However, even for 36 folders it starts to become tedious, and if this analysis needs to be expanded to feature hundreds of folders it would be completely unfeasible. Therefore we build this reusable bit of code.

In [72]:
### CHECK THAT EVERY FOLDER CONTAINS THE FOUR FILES WE NEED ### 

''' 
This function writes every csv file into a list, which we can then 
check contains all the data we need. 

To write this code I simply checkedin each month folder to see if there
was a document that contained the name of each of the police forces. 
Thankfully the names were fairly consistent and didnt contain random 
white space or typos which would have made my job alot harder.

'''
def find_crime_files(base_dir: Path, forces: list[str]):
    files = []
    for month_folder in sorted(base_dir.iterdir()): # we sort all of the folders at ingestion so the data is already in order 
    # the code above means that if anything other than a folder containing csvs enters our Path we would be in trouble
        for csv_file in month_folder.iterdir(): # again if anything other than a csv is present we would be in trouble
            if any(force in csv_file.name for force in forces): # checks if the name contains a string from the list of police stations we defined earlier 
                files.append(csv_file)
    return files

crime_data = find_crime_files(data_directory, police_forces)
print(f'Found {len(crime_data)} files')


Found 144 files


Again, we found the right number of files. Our approach did not confirm that they are all correct, but this is fine for now because the data came directly from the government website, however if the project were expanded in the future it would be worthwhile building in safeguards.

The next step would be to take a look at a few files individually.  

In [73]:
sample = pd.read_csv(crime_data[0])
print(sample.shape)
print(sample.columns.tolist())

print(sample['LSOA code'].isna().value_counts())

sample.head(6)



(105947, 12)
['Crime ID', 'Month', 'Reported by', 'Falls within', 'Longitude', 'Latitude', 'Location', 'LSOA code', 'LSOA name', 'Crime type', 'Last outcome category', 'Context']
LSOA code
False    104127
True       1820
Name: count, dtype: int64


,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,a5a2ab72258bfb2e808347707bcc312470d031825e0911...,2023-06,Metropolitan Police Service,Metropolitan Police Service,-0.685028,50.780596,On or near Park Road,E01031437,Arun 017E,Violence and sexual offences,Status update unavailable,NaN
1,16efc0a615f7368d0d34c82d9989a1eb485616f8bc56d3...,2023-06,Metropolitan Police Service,Metropolitan Police Service,-0.686514,50.780694,On or near Stocker Road,E01031437,Arun 017E,Violence and sexual offences,Status update unavailable,NaN
2,8400c77980478215c845adbe62aae9b205385268ca8b4a...,2023-06,Metropolitan Police Service,Metropolitan Police Service,0.876053,51.171725,On or near Hereford Close,E01032810,Ashford 001F,Violence and sexual offences,Investigation complete; no suspect identified,NaN
3,f087d98aff683f2a8e59de5cf0ae87c876dd77794fd804...,2023-06,Metropolitan Police Service,Metropolitan Police Service,0.870748,51.148056,On or near Bank Street,E01034986,Ashford 005G,Violence and sexual offences,Status update unavailable,NaN
4,712a91f234afbf09abed63e2835b9749632d5fd4af8e27...,2023-06,Metropolitan Police Service,Metropolitan Police Service,0.138830,51.583433,On or near Thatches Grove,E01000027,Barking and Dagenham 001A,Public order,Unable to prosecute suspect,NaN
5,NaN,2023-06,Metropolitan Police Service,Metropolitan Police Service,0.138830,51.583433,On or near Thatches Grove,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN


In [74]:
'''

Code to check whether columns are consistent across all of the different
datasets. 

'''

first_columns = pd.read_csv(crime_data[0]).columns.tolist()
mismatches = []

for file in crime_data:
    cols = pd.read_csv(file).columns.tolist()
    if cols != first_columns:
        mismatches.append((file.name, cols))

print(f'Checked {len(crime_data)} files')
print(f'Mismatches found: {len(mismatches)}')

for name, cols in mismatches:
    print(name, cols)

Checked 144 files
Mismatches found: 0


Worth nothing that we checked EVERY file in that approach. However, if our dataset gets larger we can write code to check a sample of files, or perhaps check any newly added files against the default. 

Another fact worth noting is that month and year are grouped in the same column 'Month' which might be worth separating in the future. 

**File Ingestion**

Its now time to build the loop that will take in all of our files. 

The brief separates ingestion and aggregation into two separate parts of our methodology, but I think its prudent to aggregate down a little bit during ingestion. I will never need data which is more granular than (LSOA x crime type x month). From this data I can fully derive my other dataset (Police force x crime type x month), and I will save alot of memory by aggregating during ingestion.

Again, for 144 files it might be a touch unnecesary, but if the intention is to build a reusuable pipeline, this step is invaluable. 

There is, however, a problem we have to note. This approach only works because the police force dataset is relatively clean, and any missing LSOA values will be records as NaNs, which we can then aggregate. I will deal with missing values by counting them and grouping them in their own category 'Unknown'. 

This approach will let me include them later in the less granular (crime type x force x month) dataset, but will also let the analysis team (which will be me later lol) know how many values didnt have an LSOA code reported. 

If I skip this step and simply group by (what I originally did), Python will simply ignore all NaNs. 

The final consideration is duplicate rows. Although this is technically data cleaning, which falls a bit later in our pipeline, I have to do it now if I want to aggregate as I ingest the data. The reason being is that aggregation to the desired grain will quietly absorb duplicates. Before I start removing rows however, Im going to take a closer look at the nature of the duplicates.


In [75]:
### Checking for duplicate rows in original data 

total_raw_rows = 0
total_exact_duplicates = 0
total_duplicate_crime_ids = 0

for file in crime_data:
    df = pd.read_csv(file)
    
    total_raw_rows += len(df)
    total_exact_duplicates += df.duplicated().sum()
    
    # separately, check duplicate Crime IDs (excluding nulls, since those are expected for ASB etc.)
    non_null_ids = df['Crime ID'].dropna()
    total_duplicate_crime_ids += non_null_ids.duplicated().sum()
    
    del df

print(f'Total raw rows checked: {total_raw_rows:,}')
print(f'Exact duplicate rows (all columns identical): {total_exact_duplicates:,}')
print(f'Duplicate Crime IDs (excluding nulls): {total_duplicate_crime_ids:,}')

Total raw rows checked: 5,201,924
Exact duplicate rows (all columns identical): 384,822
Duplicate Crime IDs (excluding nulls): 33,671


In [76]:
duplicate_crime_types = []

for file in crime_data:
    df = pd.read_csv(file)
    dupes = df[df.duplicated(keep=False)]
    if not dupes.empty:
        duplicate_crime_types.append(dupes['Crime type'].value_counts())
    del df

combined_dupe_types = pd.concat(duplicate_crime_types).groupby(level=0).sum()
print(combined_dupe_types.sort_values(ascending=False))

Crime type
Anti-social behaviour           555729
Violence and sexual offences      4489
Public order                      2432
Drugs                             1127
Robbery                            357
Criminal damage and arson          195
Burglary                           161
Vehicle crime                       82
Theft from the person               75
Other theft                         70
Other crime                         55
Shoplifting                         49
Possession of weapons               41
Bicycle theft                        6
Name: count, dtype: int64


In [77]:
sample_file = crime_data[0]  # pick one file to inspect, adjust as needed
df = pd.read_csv(sample_file)


# ASB duplicates (expected, no Crime ID)
asb_dupes = df[df.duplicated(keep=False) & df['Crime type'].eq('Anti-social behaviour')]
print('Sample ASB duplicates (no Crime ID):')
asb_dupes.head(10)


Sample ASB duplicates (no Crime ID):


,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
5,NaN,2023-06,Metropolitan Police Service,Metropolitan Police Service,0.138830,51.583433,On or near Thatches Grove,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN
7,NaN,2023-06,Metropolitan Police Service,Metropolitan Police Service,0.140194,51.582356,On or near Hatch Grove,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN
9,NaN,2023-06,Metropolitan Police Service,Metropolitan Police Service,0.138830,51.583433,On or near Thatches Grove,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN
10,NaN,2023-06,Metropolitan Police Service,Metropolitan Police Service,0.135924,51.587353,On or near Gibbfield Close,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN
11,NaN,2023-06,Metropolitan Police Service,Metropolitan Police Service,0.135924,51.587353,On or near Gibbfield Close,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN
12,NaN,2023-06,Metropolitan Police Service,Metropolitan Police Service,0.140194,51.582356,On or near Hatch Grove,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN
48,NaN,2023-06,Metropolitan Police Service,Metropolitan Police Service,0.138643,51.578526,On or near Geneva Gardens,E01000029,Barking and Dagenham 001C,Anti-social behaviour,NaN,NaN
49,NaN,2023-06,Metropolitan Police Service,Metropolitan Police Service,0.138643,51.578526,On or near Geneva Gardens,E01000029,Barking and Dagenham 001C,Anti-social behaviour,NaN,NaN
50,NaN,2023-06,Metropolitan Police Service,Metropolitan Police Service,0.139552,51.579445,On or near Yew Tree Gardens,E01000029,Barking and Dagenham 001C,Anti-social behaviour,NaN,NaN
51,NaN,2023-06,Metropolitan Police Service,Metropolitan Police Service,0.133322,51.579567,On or near Tolworth Gardens,E01000029,Barking and Dagenham 001C,Anti-social behaviour,NaN,NaN


This shows us that while we do have some genuine duplicates, many of our duplicates do not have crime IDS. We can infer this because we would expect the number of exact duplicate rows to be smaller than or equal to the number of duplicated Crime IDs, however the exact duplicates FAR exceed the count of duplicated IDS. The only situation where that would make sense is if most of our duplicate rows have NaNs as their crime ID. 

This now poses an interesting question regarding what to do with these duplicate values. It is clear that we should remove any duplicated crime IDs since they are data entry mistakes, however it is completely possible for a crime to be committed without a crime ID being recorded, and anothe crime with the exact same values for everything else to be committed in the same month in the same area. 

This is especially likely because of the snapping mechanism that they use to detirmine latitude and longtitude. The documentation reads:

'We maintain a master list of anonymous map points. Each map point is specifically chosen so that it: Appears over the centre point of a street, above a public place such as a Park or Airport, or above a commercial premise like a Shopping Centre or Nightclub. Has a catchment area which contains at least eight postal addresses or no postal addresses at all.

When crime data is uploaded by police forces, the exact location of each crime is compared against this master list to find the nearest map point. The co-ordinates of the actual crime are then replaced with the co-ordinates of the map point. If the nearest map point is more than 20km away, the co-ordinates are zeroed out.'

And the source is here (https://data.police.uk/about/).

Moreover, the vast majority of the duplicates are anti-social behaviour, which is very likely to be repeat offenders (loud house parties, noise complaints, etc) or have several complaints from the same area. 

Given all of this evidence, I find it very hard to simply remove all of this data, particularly because it could lead to under representation of anti-social behaviour in any crime statistics. Instead, I will remove any true duplicates, where they duplicate crime ID, and when crime ID is unknown I will leave the data point alone. 

This does mean that some actual duplicates will be missed, but I think this solution is going to give more accurate analysis in the future given how large a percentage of anti-social behaviour likely appear in the crimes we would be removing. 

In [78]:
### Ingestion Step ### 

#initiating stats and list to keep track of files
reduced_dfs = []
stats = {

    'total_rows_in': 0,
    'csv_files_processed': 0,
    'missing_lsoa_rows': 0,
    'true_duplicates_removed' : 0

}

# function to remove duplicates (defined as having duplicate IDs)

def remove_true_duplicates(df):
    
    # removing duplicates only for the subset of data which has non NULL ids

    id_dupe_removed = df[df['Crime ID'].notna()].drop_duplicates(
        subset = ['Crime ID'])
    without_id = df[~(df['Crime ID'].notna())]

    # returning the coombined df
    return pd.concat([id_dupe_removed, without_id], ignore_index = True)



# ingestion loop

for file in crime_data:
    
    #reading in file
    df = pd.read_csv(file)

    # updating initial stats
    stats['total_rows_in'] += len(df)
    stats['csv_files_processed'] += 1

    # removing duplicate IDs
    rows_before_dupe_remove = len(df)
    df = remove_true_duplicates(df)
    stats['true_duplicates_removed'] += rows_before_dupe_remove - len(df)

    # checking for missing LSOA values and recording the value
    missing_lsoa = df['LSOA code'].isna().sum()
    stats['missing_lsoa_rows'] += missing_lsoa

    # replace nans with 'Unknown' 
    df['LSOA code'].fillna('Unknown', inplace= True)


    # aggregation step
    # reducing down to (LSOA x crime type x month) 
    reduced = (
        df.groupby(['Falls within','LSOA code', 'Month', 'Crime type']).size().reset_index(name = 'crime_count')
    )

    # appending to list of reduced dataframes
    reduced_dfs.append(reduced)
    del df

ingested_data = pd.concat(reduced_dfs, ignore_index = True)

### AI GENERATED CODE ###

'''

Prompted the AI to write me some tests to check whether my aggregation
worked as intended. 

'''

print(f'Files processed: {stats['csv_files_processed']}')
print(f'Total raw rows read in: {stats['total_rows_in']:,}')
print(f'True duplicates removed: {stats['true_duplicates_removed']:,}')
print(f'Rows with missing LSOA code: {stats['missing_lsoa_rows']:,}')
print(f'Rows after LSOA-level reduction: {len(ingested_data):,}')

# sanity check: total crime_count after aggregation should equal total raw
# rows read in, since aggregation should only ever regroup rows, never drop them

### AI GENERATED CODE ###


C:\Users\hrist\AppData\Local\Temp\ipykernel_34092\2544034689.py:50: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['LSOA code'].fillna('Unknown', inplace= True)
C:\Users\hrist\AppData\Local\Temp\ipykernel_34092\2544034689.py:50: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For e

Files processed: 144
Total raw rows read in: 5,201,924
True duplicates removed: 33,671
Rows with missing LSOA code: 19,756
Rows after LSOA-level reduction: 1,718,865


We've completed ingestion of the police files at the (LSOA x crime type x month) grain and ran some sanity checks to make sure that the data is preserved.

The next step is to bring in the enrichment datasets. We could do this later, but for my sanity I would prefer to keep all ingestion in this first section so that its easier to trace potential errors and track where everything is happning.

These datasets are all much smaller snapshots and should be much simpler to ingest 

### Enrichment Datsets

In [79]:
### INGESTING IMD DATA ###

imd_path = Path('Enrichment Datasets/Index of multiple deprivations.csv')  

imd_data = pd.read_csv(imd_path)


In [80]:
# Having a peak at the data
print(f'Dimension: {imd_data.shape}')
print(f'Columns: {imd_data.columns.tolist()}')
imd_data.describe()

Dimension: (33755, 6)
Columns: ['LSOA code (2021)', 'LSOA name (2021)', 'Local Authority District code (2024)', 'Local Authority District name (2024)', 'Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)', 'Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)']


,Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived),Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)
count,33755.000000,33755.000000
mean,16878.000000,5.500074
std,9744.373505,2.872324
min,1.000000,1.000000
25%,8439.500000,3.000000
50%,16878.000000,6.000000
75%,25316.500000,8.000000
max,33755.000000,10.000000


Interestingly, the dataset contains two separate measures, the first one is the ranking of every LSOA, and the second is a score of 1 to 10. The column description says that 1 is the top 10% most deprived LSOAs.

This gives us two separate measures to potentially use and I shall have to put some more thought into deciding which one to use. 


#### ASK IN QNA which one to use - i prefer decile bcs interpretability


In [81]:
### INGESTING POPULATION DATA ###

population_path = Path('Enrichment Datasets/LSOA population data.xlsx')

# reading in an excel file is slightly different than a typical csv
population_data = pd.read_excel(
    population_path, sheet_name = 'Mid-2024 LSOA 2021',
    skiprows = 3 # some blank rows at the top of the sheet, took some trial and error to find how many to skip until we get to the real titles
)

# sanity check, making sure our LSOA numbers line up

print(f'Population file rows: {len(population_data):,}')
print(f'IMD file rows: {len(imd_data):,}')
print(f'Difference: {len(population_data) - len(imd_data):,}')

Population file rows: 35,672
IMD file rows: 33,755
Difference: 1,917


Here I was a bit worried about data quality because we have a clear difference in the number of LSOAs. However I believe that this can be chalked up to our IMD files not containing any Welsh LSOAs, which should acount for the difference. 

Checking the official documentation, this appears to be the case, but its useful to confirm ourselves.

The code cell below runs some tests to investigate if this is the case. We count the number of LSOA codes in the population file which start with W.

In [82]:
population_welsh = population_data[population_data['LSOA 2021 Code'].str.startswith('W')]
print(len(population_welsh))

1917


Indeed, this is exactly the difference we identified earlier. Now since all four of our chosen police forces are entirely contained in England, this wont be a problem. It might even be prudent to remove all of the welsh codes from th population dataset, but in the interest of sticking closely to the brief's suggested data pipeline I elect not to do that now - keeping the ingestion phase mostly relegated to doing one job.

In [83]:
#Taking a peak at the data

print(f'Shape: {population_data.shape}')
population_data.columns.tolist()

Shape: (35672, 15)


['LAD 2021 Code',
 'LAD 2021 Name',
 'LSOA 2021 Code',
 'LSOA 2021 Name',
 'Total',
 'F0 to 15',
 'F16 to 29',
 'F30 to 44',
 'F45 to 64',
 'F65 and over',
 'M0 to 15',
 'M16 to 29',
 'M30 to 44',
 'M45 to 64',
 'M65 and over']

We can now move on to importing the final dataset- LSOA boundaries. This is a fun one because it will allow us to make maps in powerbi later (hopefully), but it is my first time importing a geojson file so it might take some finagling. 

Upon doing some research, I found that importing such a file can be done most simply by using a package called *geopandas*. geojson files work similarly to normal pandas dfs when imported except with an extra 'geometry' column which contains the shapes for mapping. 



In [84]:
### IMPORTING GEOGRAPHIC DATA ### 

geography_path = Path('Enrichment Datasets/Lower_layer_Super_Output_Areas_December_2021.geojson')

geography_data = gpd.read_file(geography_path)


In [85]:
# Some basic checks

print(f'Shape: {geography_data.shape}')
geography_data.columns.tolist()

Shape: (35672, 10)


['FID',
 'LSOA21CD',
 'LSOA21NM',
 'LSOA21NMW',
 'BNG_E',
 'BNG_N',
 'LAT',
 'LONG',
 'GlobalID',
 'geometry']

Awesome, we can see that the row count matches that of the population data which contains both England and Wales, nothing to be alarmed about here. 

The geometry column is there as expected, but deciphering the meaning of the column names might be a bit tricky. I referenced official documentation and found that the columns represent the following:

- `LSOA21CD` — LSOA code (join key)
- `LSOA21NM` — LSOA name
- `LSOA21NMW` — LSOA name in Welsh (blank for English LSOAs)
- `BNG_E` / `BNG_N` — British National Grid coordinates 
- `LAT` / `LONG` — latitude/longitude of the LSOA's centre point
- `GlobalID` / `FID` — internal system IDs

With this file done we now have all of our datasets loaded, and we can move on to do some data cleaning and validation.


## Data Cleaning and Validation

The brief offers guidance for how to approach this step. I will follow the suggested protocol and deviate where I think neccesary.

A good place to start is with a thorough data quality check. Practically, this means that we should check unique values, look for casing/spelling inconsistencies, examine null counts for different columns. This should be done for all of our datasets, but it is most relevant for the police one because there are categories such as crime type which are very prone to these types of errors. 


In [86]:
## Crime dataset data quality assesment ## 

# shape and dtypes

print('Shape:', ingested_data.shape)
print('\nColumn types:')
print(ingested_data.dtypes)

Shape: (1718865, 5)

Column types:
Falls within    object
LSOA code       object
Month           object
Crime type      object
crime_count      int64
dtype: object


In [87]:
# nulls analysis

print(f'Nulls per column: {ingested_data.isna().sum()}')

Nulls per column: Falls within    0
LSOA code       0
Month           0
Crime type      0
crime_count     0
dtype: int64


This is actually remarkable news, there are 0 missing values in any of our columns. Although I say remarkable, this is partly due to the little bit of data cleaning we performed on entry to represent missing LSOA values as 'Unknown'.  

In [88]:
'''

Some sanity checks:

- We should have precisely 36 unique 'Months' for now because month is 
    currently still a year-month hybrid.

- We should have a few thousand LSOAs 

- We should have a manageable list of crime types. Indeed, if its a very
    long list we will have alot of data entry issues. Hopefully the 
    police force use a small finite list of crime types and their 
    database rejects/fixes capitalisation issues at data entry.

'''

print(f'Unique LSOA codes: {ingested_data['LSOA code'].nunique()}')
print(f'Unique dates: {ingested_data['Month'].nunique()}')
print(f'Unique Crime Types: {ingested_data['Crime type'].nunique()}')

Unique LSOA codes: 13996
Unique dates: 36
Unique Crime Types: 14


Okay some interesting results here. Firstly we have precisely 36 'Months' which is good, and a very small list of crime types suggesting that the column is already cleaned. 

The potential problems come with the numberof unique LSOA codes. 13,996 codes is more than I was expecting, given that there are 33,755 in the entirety of England and we are only looking at four police forces, I was expecting to have far fewer. 

This is certainly something worth investigating and I will do so below. 

In [89]:
'''

Initially I thought this check would be simple, but it turns out to be
more complicated than I thought. 

It is not possible to simply count the unique LSOA codes in the original
datasets because an LSOA that has crimes in every month could 
theoretically appear 36 times. 

Instead my idea is to concatanate every single appearance of an LSOA code
from the original datasets into a single really long list and then let 
Pandas do the heavy lifting of figuring out which ones are unique. This
would confirm the count, but I also think its important to know how 
many of our LSOAs are coming from each police force, so I adjusted the 
code to keep track of which force each code came from aswell.

'''

records = {} # initiating empty dictionary to hold pairs of forces and codes

# adding all of the LSOA codes to a dictionary, with their respective police forces
for file in crime_data:
    df = pd.read_csv(file, usecols = ['LSOA code'])
    # below is a lovely line of code not written by me
    # AI came up with this 
    # The idea is that it goes through all of our police forces, checking
    # whether they appear in the file name. If it finds a match, the loop
    # breaks, and the corresponding LSOA codes will be assigne to this 
    # police force.
    corresponding_force = next((force for force in police_forces if force in file.name), None)

    # the first time we see a force, initialise an empty list for it
    if corresponding_force not in records:
        records[corresponding_force] = []

    # add all of this files LSOA codes to its force's list
    records[corresponding_force].extend(df['LSOA code'].dropna().tolist())
    del df # stop storing this df

# detirmining which LSOA codes are unique
for force, codes in records.items():
    # have to convert to series object first
    unique_count = pd.Series(codes).nunique()
    print(f' The {force} has {unique_count} unique LSOAs')

# counting the total unique LSOAs
total_LSOAs = sum(pd.Series(codes).nunique() for force, codes in records.items())
print(f'Unique LSOAs from original data: {total_LSOAs}')
print(f'Unique LSOAs in aggregated data: {ingested_data['LSOA code'].nunique()}')

 The metropolitan has 11389 unique LSOAs
 The northumbria has 1056 unique LSOAs
 The surrey has 855 unique LSOAs
 The west-midlands has 1760 unique LSOAs
Unique LSOAs from original data: 15060
Unique LSOAs in aggregated data: 13996


In [90]:
met_lsoas = pd.Series(list(set(records['metropolitan'])), name='LSOA code')

met_lsoas_named = met_lsoas.to_frame().merge(
    geography_data[['LSOA21CD', 'LSOA21NM']],
    left_on='LSOA code',
    right_on='LSOA21CD',
    how='left'
)

print(met_lsoas_named['LSOA21NM'].isna().sum(), 'LSOA codes with no match in boundary file')

# check how many distinct place-name prefixes show up
met_lsoas_named['area_name'] = met_lsoas_named['LSOA21NM'].str.rsplit(' ', n=1).str[0]
print(met_lsoas_named['area_name'].nunique(), 'distinct area names')
print(met_lsoas_named['area_name'].value_counts().head(40))

0 LSOA codes with no match in boundary file
330 distinct area names
area_name
Croydon                   228
Barnet                    220
Birmingham                204
Ealing                    199
Bromley                   199
Wandsworth                186
Enfield                   183
Newham                    182
Brent                     181
Lambeth                   181
Lewisham                  175
Southwark                 173
Hillingdon                170
Buckinghamshire           167
Redbridge                 163
Greenwich                 163
Tower Hamlets             163
Havering                  153
Hounslow                  150
Hackney                   149
Bexley                    147
Haringey                  147
Waltham Forest            146
Harrow                    144
Camden                    130
Islington                 126
Merton                    126
Westminster               123
Sutton                    123
Barking and Dagenham      115
Richmond upon Thames  

In [91]:
london_boroughs = [
    'Croydon', 'Barnet', 'Ealing', 'Bromley', 'Wandsworth', 'Enfield', 'Newham',
    'Brent', 'Lambeth', 'Lewisham', 'Southwark', 'Hillingdon', 'Redbridge',
    'Greenwich', 'Tower Hamlets', 'Havering', 'Hounslow', 'Hackney', 'Haringey',
    'Bexley', 'Waltham Forest', 'Harrow', 'Camden', 'Merton', 'Islington',
    'Sutton', 'Westminster', 'Richmond upon Thames', 'Barking and Dagenham',
    'Hammersmith and Fulham', 'Kensington and Chelsea', 'Kingston upon Thames',
    'City of London'
]

# going to use sets to isolate unique codes

# codes that are within london boroughous 
london_lsoas = set(
    met_lsoas_named[met_lsoas_named['area_name'].isin(london_boroughs)]['LSOA code']
)

# codes that seem to fall outside of london boroughs
out_of_area_lsoas = set(
    met_lsoas_named[~met_lsoas_named['area_name'].isin(london_boroughs)]['LSOA code']
)

print(f'Met area LSOAs: {len(london_lsoas)}')
print(f'Out of Met area LSOAs: {len(out_of_area_lsoas)}')

Met area LSOAs: 4981
Out of Met area LSOAs: 6408


In [92]:
# needed some help from AI here, I accidentally deleted a cell earlier
# couldnt recover it and forgot how I derived the variable 'met_files'
# trying to do it again led to disaster after disaster

met_files = []

root = Path('Full June 2023 - May 2026 Police datasets')


for folder in root.iterdir():
    if folder.is_dir():
        met_file = folder / f'{folder.name}-metropolitan-street.csv'
        if met_file.exists():
            met_files.append(met_file)

print(f'Found {len(met_files)} files')

Found 36 files


In [93]:
'''

Trying to figure out what percentage of all crimes seem to appear outside
of the Met polices jurisdiction

'''

# initiating empty lists to count crimes
total_crimes = 0
out_of_area_crimes = 0

# looping through all of my files and appending all crimes to the total_crimes list
# and appending crimes that fall outside of the met's jurisdiction 
# to their own list 
for file in met_files:
    df = pd.read_csv(file, usecols=['LSOA code'])
    total_crimes += len(df)
    out_of_area_crimes += df['LSOA code'].isin(out_of_area_lsoas).sum()
    del df


# printing out results
print(f'Total Met crimes: {total_crimes:,}')
print(f'Out of met crimes: {out_of_area_crimes:,}')
print(f'Percentage: {(out_of_area_crimes / total_crimes) * 100}%')

Total Met crimes: 3,420,471
Out of met crimes: 14,165
Percentage: 0.4141242536481087%


Okay that concludes our investigation. So my suspicions that the LSOA count for the metropolitan police might be far too high were absolutely justified. 

A whopping 6408 of the LSOA codes which appear in the dataset have a corresponding LSOA area name which falls outside of the metropolitan police jurisdiction. Despite this, only 0.4% of the total crimes are reported in these areas. 

After consulting with Miguel, I decided that the best approach would be to simply note down the LSOA code as 'out-of-bounds'. That way we don't remove data which can be used in the Force x Crime type x month grain, but can exclude it from the LSOA x crime type x month grain, which actually relies on knowing the LSOA for it to be useful.

Another point worth mentioning is that in my investigation I cross checked all of the LSOA code area names against a list of London boroughs, in practice, I will find a lookup table which maps police force -> LSOA code, and filter my entire dataset with that. This is for two reasons: this data error can also have occurred in the other four datasets, and I have not checked those, and also it would simply be more accurate. 

I will perform the merge now, and then carry on with data cleaning and validation for the merged dataset. That way I can check for any unintentional duplicates. 

In [94]:
# loading in the lookup table

pfa_lookup_path = Path('Enrichment Datasets/LSOA to Police lookup.csv')

pfa_lookup = pd.read_csv(pfa_lookup_path)


In [95]:
# taking a look at the data

print(pfa_lookup.head(10))

ingested_data.head(10)

     LAD24CD         LAD24NM    CSP24CD         CSP24NM    PFA24CD  \
0  E07000136          Boston  E22000173          Boston  E23000020   
1  E07000137    East Lindsey  E22000174    East Lindsey  E23000020   
2  E07000138         Lincoln  E22000175         Lincoln  E23000020   
3  E07000139  North Kesteven  E22000176  North Kesteven  E23000020   
4  E07000140   South Holland  E22000177   South Holland  E23000020   
5  E07000141  South Kesteven  E22000178  South Kesteven  E23000020   
6  E07000142    West Lindsey  E22000179    West Lindsey  E23000020   
7  E07000143       Breckland  E22000218       Breckland  E23000024   
8  E07000144       Broadland  E22000219       Broadland  E23000024   
9  E07000145  Great Yarmouth  E22000220  Great Yarmouth  E23000024   

        PFA24NM  ObjectId  
0  Lincolnshire         1  
1  Lincolnshire         2  
2  Lincolnshire         3  
3  Lincolnshire         4  
4  Lincolnshire         5  
5  Lincolnshire         6  
6  Lincolnshire         7  
7    

,Falls within,LSOA code,Month,Crime type,crime_count
0,Metropolitan Police Service,E01000001,2023-06,Criminal damage and arson,1
1,Metropolitan Police Service,E01000001,2023-06,Other theft,1
2,Metropolitan Police Service,E01000002,2023-06,Criminal damage and arson,2
3,Metropolitan Police Service,E01000002,2023-06,Shoplifting,1
4,Metropolitan Police Service,E01000002,2023-06,Theft from the person,2
5,Metropolitan Police Service,E01000003,2023-06,Burglary,1
6,Metropolitan Police Service,E01000003,2023-06,Vehicle crime,1
7,Metropolitan Police Service,E01000005,2023-06,Anti-social behaviour,1
8,Metropolitan Police Service,E01000005,2023-06,Burglary,1
9,Metropolitan Police Service,E01000005,2023-06,Other theft,3


Taking a look at the data reveals some confuding column names. Consulting the official documentation shows that only the following columns are relevant to our merge:

- `LAD24CD` - Local Authority District code

- `LAD24NM` - Local Authority District name

- `PFA24CD` - Police Force Area Code

- `PFA24NM` - Police force Area name  

We only need PFA24NM and LAD24CD to performour merge. However we can imemdiately see a naming mismatch. In our crime datset each police districts name is written as 'District Name Police', whereas this dataset eshews the word 'Police'. This is something that we can resolve before our merge, but first I take a bit of a closer look at the data.


In [96]:
# looking at unique police force names in lookup table

print(pfa_lookup['PFA24NM'].unique())

# comparing against the polie force data

print('')
print('-------')
print('')

print(ingested_data['Falls within'].unique())  

['Lincolnshire' 'Norfolk' 'Nottinghamshire' 'Thames Valley'
 'Staffordshire' 'Suffolk' 'Surrey' 'Warwickshire' 'Sussex' 'Cleveland'
 'Durham' 'Cheshire' 'Lancashire' 'Humberside' 'North Yorkshire'
 'Derbyshire' 'Leicestershire' 'West Mercia' 'Avon and Somerset'
 'Devon & Cornwall' 'Wiltshire' 'Cambridgeshire' 'Bedfordshire' 'Essex'
 'Kent' 'Hampshire' 'Northumbria' 'Dorset' 'Gloucestershire'
 'Hertfordshire' 'Northamptonshire' 'Cumbria' 'Greater Manchester'
 'Merseyside' 'South Yorkshire' 'West Midlands' 'West Yorkshire'
 'London, City of' 'Metropolitan Police' 'North Wales' 'Dyfed-Powys'
 'South Wales' 'Gwent']

-------

['Metropolitan Police Service' 'Northumbria Police' 'Surrey Police'
 'West Midlands Police']


The differences are as discussed, with the exception of Metopolitan Police which has an extra 'Service' added on. We can either filter as is, or change one of the datasets to match the other. 

I opt to change the lookup table's names using a simple dictionary approach because the police dataset is our main datset and I see no reason to change its naming convention.

In [97]:
# creating a dictionary to change our four police forces

force_name_mapping = {
    'Metropolitan Police': 'Metropolitan Police Service',
    'Northumbria': 'Northumbria Police',
    'Surrey': 'Surrey Police',
    'West Midlands': 'West Midlands Police',
}

# replacing the values in the lookup table
pfa_lookup['PFA24NM'] = pfa_lookup['PFA24NM'].replace(force_name_mapping)

# having a look to see if it worked
print(pfa_lookup['PFA24NM'].unique())


['Lincolnshire' 'Norfolk' 'Nottinghamshire' 'Thames Valley'
 'Staffordshire' 'Suffolk' 'Surrey Police' 'Warwickshire' 'Sussex'
 'Cleveland' 'Durham' 'Cheshire' 'Lancashire' 'Humberside'
 'North Yorkshire' 'Derbyshire' 'Leicestershire' 'West Mercia'
 'Avon and Somerset' 'Devon & Cornwall' 'Wiltshire' 'Cambridgeshire'
 'Bedfordshire' 'Essex' 'Kent' 'Hampshire' 'Northumbria Police' 'Dorset'
 'Gloucestershire' 'Hertfordshire' 'Northamptonshire' 'Cumbria'
 'Greater Manchester' 'Merseyside' 'South Yorkshire'
 'West Midlands Police' 'West Yorkshire' 'London, City of'
 'Metropolitan Police Service' 'North Wales' 'Dyfed-Powys' 'South Wales'
 'Gwent']


Although not immediately obvious, its possible to find the four police forces that we need with their names changed in the list above. 

Now its a case of performing the merge.

Its important to break this down clearly, since this merge requires chaining two datasets. Our lookup table contains LAD and police force, but our main dataset contains LSOA and police force. This means that its not yet possible to join them directly. 

Instead, we can use the population data, which has both LAD and LSOA codes to join to our lookup table, thus homebrewing our own LSOA -> Police force lookup table, which will then be applied to the original dataset.

I perform this chain below, checking the resulting dataframes at each step to ensure correctness.



In [98]:
'''

First I look at the population file, I need to make sure that each LSOA
code maps to precisely one LAD. If that isnt the case then we may 
introduce unintentional duplicates.

'''

lsoa_to_lad = population_data[['LSOA 2021 Code', 'LAD 2021 Code']].drop_duplicates()

duplicate_lsoa = lsoa_to_lad['LSOA 2021 Code'].duplicated().sum()
print(duplicate_lsoa)


0


Having found 0 duplicated LSOA codes, we can now continue.

In [99]:
'''

The next step is to chain the LSOA codes to th PFA lookup table, creating
a PFA -> LSOA lookup table. This is functionally a simple left join. 

'''

lsoa_to_pfa = lsoa_to_lad.merge(
    pfa_lookup[['LAD24CD', 'PFA24NM']], # these are the only two columns we need as aforementione
    left_on ='LAD 2021 Code',
    right_on = 'LAD24CD',
    how ='left'
)

In [100]:
# taking a look at the resulting table

print(lsoa_to_pfa.head())

# checking to see if any rows are unmatched 
print(f'Rows with no PFA match: {lsoa_to_pfa['PFA24NM'].isna().sum()}')

# checking whether the number of rows changed at all
print(f'Rows before merge: {len(lsoa_to_lad)}')
print(f'Rows after merge: {len(lsoa_to_pfa)}')


  LSOA 2021 Code LAD 2021 Code    LAD24CD    PFA24NM
0      E01011949     E06000001  E06000001  Cleveland
1      E01011950     E06000001  E06000001  Cleveland
2      E01011951     E06000001  E06000001  Cleveland
3      E01011952     E06000001  E06000001  Cleveland
4      E01011953     E06000001  E06000001  Cleveland
Rows with no PFA match: 0
Rows before merge: 35672
Rows after merge: 38790


Okay, we found some duplicates, this means that there were likely some duplicated LAD values in the pfa_lookup table. The easiest way I can think to fix this is to simply de-duplicate the data and perform the merge. 

Ordinarily I would go back and fix it, but I have left this here as evidence of my process.

In [101]:
# dropping duplicated LAD values
lad_to_pfa = pfa_lookup[['LAD24CD', 'PFA24NM']].drop_duplicates()

# now both of the tables we need should be un-duplicated

# performing the merge again

lsoa_to_pfa = lsoa_to_lad.merge(
    lad_to_pfa,
    left_on='LAD 2021 Code',
    right_on='LAD24CD',
    how='left'
)

print(f'Rows before merge: {len(lsoa_to_lad)}')
print(f'Rows after merge: {len(lsoa_to_pfa)}')


Rows before merge: 35672
Rows after merge: 35672


Awesome, this has fixed our duplicate problem.

In [102]:
lsoa_to_pfa.head()

,LSOA 2021 Code,LAD 2021 Code,LAD24CD,PFA24NM
0,E01011949,E06000001,E06000001,Cleveland
1,E01011950,E06000001,E06000001,Cleveland
2,E01011951,E06000001,E06000001,Cleveland
3,E01011952,E06000001,E06000001,Cleveland
4,E01011953,E06000001,E06000001,Cleveland


We only need the LSOA and PFA columns from that table.

In [103]:
lsoa_to_pfa = lsoa_to_pfa[['LSOA 2021 Code', 'PFA24NM']]

lsoa_to_pfa.head()

,LSOA 2021 Code,PFA24NM
0,E01011949,Cleveland
1,E01011950,Cleveland
2,E01011951,Cleveland
3,E01011952,Cleveland
4,E01011953,Cleveland


We now have the neccesary lookup table, which we can merge with our police data.

In [104]:
crime_data_location_fixed = ingested_data.merge(
    lsoa_to_pfa,
    left_on = 'LSOA code',
    right_on = 'LSOA 2021 Code',
    how = 'left'
)

crime_data_location_fixed['jurisdiction_mismatch'] = (
    crime_data_location_fixed['Falls within'] != crime_data_location_fixed['PFA24NM']
)


print(f'Rows with jurisdiction mismatch: {crime_data_location_fixed['jurisdiction_mismatch'].sum()}')
print(f'As a % of total: {crime_data_location_fixed['jurisdiction_mismatch'].mean():.3%}')



Rows with jurisdiction mismatch: 16878
As a % of total: 0.982%


Now we've merged the data with our lookup table and we can see that about 1% of all of the crimes have issues with where they're reported. Before we continue any further, I will perform a few sanity checks to make sure my merge hasnt introduced any new data problems.

In [105]:
print(f'Rows before merge: {len(ingested_data)}')
print(f'Rows after merge: {len(crime_data_location_fixed)}')
print(f'Rows with no PFA match: {crime_data_location_fixed['PFA24NM'].isna().sum()}')
crime_data_location_fixed.head()

Rows before merge: 1718865
Rows after merge: 1718865
Rows with no PFA match: 563


,Falls within,LSOA code,Month,Crime type,crime_count,LSOA 2021 Code,PFA24NM,jurisdiction_mismatch
0,Metropolitan Police Service,E01000001,2023-06,Criminal damage and arson,1,E01000001,"London, City of",True
1,Metropolitan Police Service,E01000001,2023-06,Other theft,1,E01000001,"London, City of",True
2,Metropolitan Police Service,E01000002,2023-06,Criminal damage and arson,2,E01000002,"London, City of",True
3,Metropolitan Police Service,E01000002,2023-06,Shoplifting,1,E01000002,"London, City of",True
4,Metropolitan Police Service,E01000002,2023-06,Theft from the person,2,E01000002,"London, City of",True


So we havent introduced any new rows which is excellent news, and we have 563 rows with no PFA match. These rows could be problematic, but I doubt it for the following reason. Recall that our ingestion step is doing a tiny bit of data cleaning under the hood: we're marking all missing LSOA codes as 'Unknown' which isnt a real LSOA code, and they would certainly not have a matching Police Force Area.

To check if this is the case, we can look a the unique LSOA values which have no corresponding polie force introduced

In [106]:
unmatched = crime_data_location_fixed[crime_data_location_fixed['PFA24NM'].isna()]
print(unmatched['LSOA code'].value_counts())


LSOA code
Unknown    563
Name: count, dtype: int64


As expected, all of these are 'Unknown'. This means that there is no problem here. 

Now the final step of this process is changing the value of the LSOA code where we have a mismatch in the 'Falls within' column and the newly joined 'PFA24NM' column. My decision from earlier was to flag all of these as unknown since they're clearly data entry issues, but I dont want to unnecesarily delete data regardless of how small a percentage it is.

I complete this final step now, before we move on with the rest of the data cleaning. 

In [107]:
# replacing all LSOA codes where jurisdiction_mismatch with 'Out of bounds''DataAnalysis_01_01_Project_PART1_Python_Data_prep-processing v1.pdf'

crime_data_location_fixed['LSOA code'] = crime_data_location_fixed['LSOA code'].where(
    ~crime_data_location_fixed['jurisdiction_mismatch'],
    'Out of bounds'
)

In [108]:
# sanity checks

print(f'Unique LSOA codes: {crime_data_location_fixed['LSOA code'].nunique()}')

lsoa_per_force = crime_data_location_fixed.groupby('Falls within')['LSOA code'].nunique()
print(lsoa_per_force)

Unique LSOA codes: 8341
Falls within
Metropolitan Police Service    4976
Northumbria Police              928
Surrey Police                   720
West Midlands Police           1720
Name: LSOA code, dtype: int64


These numbers look much better than before, especially the Met's numbers which dropped a considerable amount. We can now move on to completing the rest of our data cleaning.

We continue with **duplicate checks**. An important note here is that we removed duplicates of the raw data during ingestion, so our duplicate checks now are going to be for our aggregate LSOA x Crime type x Month Grain.

A duplicate at this point is simply when the entire row repeats, which hopefully shouldnt happen at all.

In [109]:
# checking for duplicates

duplicates = crime_data_location_fixed[
    crime_data_location_fixed.duplicated(keep=False)
]

print(f'Number of duplicate rows: {len(duplicates)}')


Number of duplicate rows: 0


Perfect, 0 duplicates. We can now move on to validating crime type.

I will check to see if all categories are consistent and what they are. 

In [110]:
# having a look at the crime type column 

crime_types = (
    crime_data_location_fixed['Crime type']
    .value_counts()
    .sort_index()
)

crime_types.sort_values(ascending = False)

Crime type
Violence and sexual offences    277962
Anti-social behaviour           224465
Vehicle crime                   186318
Other theft                     164349
Criminal damage and arson       157101
Public order                    142092
Burglary                        133483
Drugs                            94318
Shoplifting                      74327
Robbery                          69803
Theft from the person            68028
Other crime                      55993
Bicycle theft                    37959
Possession of weapons            32667
Name: count, dtype: int64

This is actually a very short list considering the size of our dataset. A couple of things to note: 

1) Bicycle theft is considered important enough to have its own category.

2) It is not clear how public order is distinguished from Anti-social behaviour.

3) It is not clear what 'other theft' includes. 

Despite these observations there is no reason to make any adjustments to this list. These classifications originate from the police data standard, and should the analysis team (me) wish to group these together in larger categories, they can do so themselves. 

The final steps would be to validate that the month column is in chronological order, and to make sure that crime count is consistent and non negative.

In [111]:
# validating month column

months = sorted(crime_data_location_fixed['Month'].unique())

print(months)
print(f'Unique months: {len(months)}')

['2023-06', '2023-07', '2023-08', '2023-09', '2023-10', '2023-11', '2023-12', '2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05']
Unique months: 36


All good on that front. Final step: crime count.

In [112]:
crime_data_location_fixed['crime_count'].describe().apply('{:,.0f}'.format)

count    1,718,865
mean             3
std              6
min              1
25%              1
50%              2
75%              3
max          1,902
Name: crime_count, dtype: object

This check confirms that crime counts are correct and there are no negatives. Everything is validated and cleaned, we can move on now to the next step inthe pipeline. 

## Feature engineering & transformation

The first step is to separate Month and Year attributes, which is easily done.

In [113]:
# first converting to a date time

crime_data_location_fixed['Month'] = pd.to_datetime(
    crime_data_location_fixed['Month'],
    format = '%Y-%m'
)

# deriving year attribute
crime_data_location_fixed['Year'] = (
    crime_data_location_fixed['Month'].dt.year
)

# month number 
crime_data_location_fixed['Month Number'] = (
    crime_data_location_fixed['Month'].dt.month
)

# month names incase we want to group by January
crime_data_location_fixed['Month Name'] = (
    crime_data_location_fixed['Month'].dt.month_name()
)

crime_data_location_fixed[
    ['Month', 'Year', 'Month Number', 'Month Name']
].head()

# we should probably remove the original 'Month' column now
crime_data_location_fixed.drop(columns= ['Month'], inplace = True)



In [114]:
crime_data_location_fixed.head()

,Falls within,LSOA code,Crime type,crime_count,LSOA 2021 Code,PFA24NM,jurisdiction_mismatch,Year,Month Number,Month Name
0,Metropolitan Police Service,Out of bounds,Criminal damage and arson,1,E01000001,"London, City of",True,2023,6,June
1,Metropolitan Police Service,Out of bounds,Other theft,1,E01000001,"London, City of",True,2023,6,June
2,Metropolitan Police Service,Out of bounds,Criminal damage and arson,2,E01000002,"London, City of",True,2023,6,June
3,Metropolitan Police Service,Out of bounds,Shoplifting,1,E01000002,"London, City of",True,2023,6,June
4,Metropolitan Police Service,Out of bounds,Theft from the person,2,E01000002,"London, City of",True,2023,6,June


Awesome, date time attributes sorted. The former Month column is also fully derivable from our existing attributes so no need to keep it.

We can now move on to the part of this step that will likely take the longest: joining our enrichment datasets. 

### Joining Enrichment data

An important part of this step is to ensure grain compatibility. However my chosen enrichment datasets should both be easily joined by a direct LSOA to LSOA join with no aggregation needed. 

Before and after every join I will check the row count to ensure no accidental duplicates have been introduced. 

#### Joining Population Dataset

In [115]:
lsoa_grain_df = crime_data_location_fixed.merge(
    population_data[['LSOA 2021 Code', 'Total']],
    left_on ='LSOA code',
    right_on = 'LSOA 2021 Code',
    how = 'left'
)

lsoa_grain_df.rename(columns = {'Total' : 'Population'}, inplace = True)

In [116]:
print(f'Rows before merge: {len(crime_data_location_fixed)}')
print(f'Rows after merge: {len(lsoa_grain_df)}')
print(f'Rows with no population match: {lsoa_grain_df['Population'].isna().sum()}')

Rows before merge: 1718865
Rows after merge: 1718865
Rows with no population match: 16878


We have some rows without a population type, but this is fine because we have some areas which do not have an LSOA code

In [117]:
lsoa_grain_df[lsoa_grain_df['LSOA code'] == 'Out of bounds'].count()

Falls within             16878
LSOA code                16878
Crime type               16878
crime_count              16878
LSOA 2021 Code_x         16315
PFA24NM                  16315
jurisdiction_mismatch    16878
Year                     16878
Month Number             16878
Month Name               16878
LSOA 2021 Code_y             0
Population                   0
dtype: int64

We can see clearly from above that the count of rows without population data is the same as the count of rows with out of bounds LSOA which makes sense.

This join went off smoothly, now we can move on to the IMD data.

#### Joining IMD dataset


In [118]:
# same process as before we use a left join and then rename the column

lsoa_grain_df = lsoa_grain_df.merge(
    imd_data[['LSOA code (2021)', 'Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)']],
    left_on = 'LSOA code',
    right_on = 'LSOA code (2021)',
    how = 'left'
)

lsoa_grain_df = lsoa_grain_df.rename( columns={'Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)': 'IMD_Decile'})

In [119]:
print(f'Rows before merge: {len(crime_data_location_fixed)}')
print(f'Rows after merge: {len(lsoa_grain_df)}')
print(f'Rows with no IMD match: {lsoa_grain_df['IMD_Decile'].isna().sum()}')

Rows before merge: 1718865
Rows after merge: 1718865
Rows with no IMD match: 16878


Same number of missing IMD values as before, so no problem here. 

Now its worth taking a look at the data and removing some columns because we will have a few duplicate columns left from the merges.

In [120]:
print(lsoa_grain_df.columns.tolist())

['Falls within', 'LSOA code', 'Crime type', 'crime_count', 'LSOA 2021 Code_x', 'PFA24NM', 'jurisdiction_mismatch', 'Year', 'Month Number', 'Month Name', 'LSOA 2021 Code_y', 'Population', 'LSOA code (2021)', 'IMD_Decile']


From this list we can see several LSOA code columns from the thre merges we performed, all of which are safe to drop. We can keep the original LSOA code column.

Ontop of that we can drop PFA24NM and jurisdiction mismatch columns since both of them were only working columns that we needed to assign 'out of bounds'  to the correct crimes. 

In [121]:
lsoa_grain_df.drop(
    columns = ['LSOA 2021 Code_x', 'LSOA 2021 Code_y', 'LSOA code (2021)',
    'PFA24NM', 'jurisdiction_mismatch'], inplace = True
)

In [122]:
print(lsoa_grain_df.columns.tolist())


['Falls within', 'LSOA code', 'Crime type', 'crime_count', 'Year', 'Month Number', 'Month Name', 'Population', 'IMD_Decile']


Our columns are now more tidy, and our datasets have been merged. Its worth noting that we introduces some nulls into the population and IMD decile columns, but that is perfectly okay because there is no way to attach a population or IMD score to an area thats out of bounds. 

These nulls are documented and known to not be a problem.

Finally, I also introduced some temporal mismatches. The population data is all from 2024, and the IMD data is a snapshot from mid 2025. This was unavoidable, and I speak in more detail about this mismatch in my documentation. 



## Aggregation for reporting

This will hopefully be a short step in comparison to the rest of my phases. I will introduce two new measures whih could be useful for the analysis. 

First I will introduce crime_rate_per_1000, a normalised crime rate. This is a super important metric for comparing crime rates in areas with different population sizes. 

Next I will introduce crime_category_share, which measures the percentage a certain crime category makes up of the total crime in an area. This is more useful than raw counts in many cases when trying to understand how crime types shift over time.  Its also another metric which is unaffected by population size. 



In [123]:
# adding crime_rate_per_1000

lsoa_grain_df['crime_rate_per_1000'] = (lsoa_grain_df['crime_count']/ lsoa_grain_df['Population']) * 1000

In [124]:
print(lsoa_grain_df['crime_rate_per_1000'].isna().sum())

16878


Same number of NANs as we expected, all is well. We can now add the nxt column. 

In [125]:
# adding category_of_share 

# first we compute the monthly totals using groupby and extracting the sums
lsoa_month_totals = (
    lsoa_grain_df.groupby(['LSOA code', 'Year', 'Month Number'])['crime_count'].transform('sum')
) 

# now we attach the percentage as a new column

lsoa_grain_df['crime_category_share'] = lsoa_grain_df['crime_count'] / lsoa_month_totals



If this was done correctly, the monthly shares should be 1. I check this below.


In [126]:

sanity_check = lsoa_grain_df.groupby(['LSOA code', 'Year', 'Month Number'])['crime_category_share'].sum()
print(sanity_check.describe())  

count    2.980490e+05
mean     1.000000e+00
std      1.220503e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: crime_category_share, dtype: float64


Perfect, this was also applied succesfully. 

With this we've essentially concluded the work we need for the LSOA x Crime type x month grain. Validation has been performed throughout to make sure that we havent introduced duplicates and that our code has been working correctly. 

So now the only thing we have left to do is to generate the second dataset grain: Police force x month x crime type. This is slightly less granular, but will save the analysis team (me) from having to perform the aggregation themselves in PowerBI. 

Given that its fairly quick and easy to do in Python there's no reason to eschew this step. 

I will derive this grain from the current dataset, with the main consideration for the aggregation step being not to vastly overcount population. Because our data currently has one row per LSOA per month, per crime type, we may count population many times over. 

To account for this I will simply deduplicate the LSOA codes so that each population is counted only once. In particular, I will have to aggregate crime count and population separately, and then merge them together.

Moreover, I couldn't find a clever way to aggregate my custom metric columns, so I simply recompute them at the end.


In [127]:
# aggregating crime counts first, this will end up being the final dataset
# sincew attach everything else to it

force_grain_df = (
    lsoa_grain_df.groupby(['Falls within', 'Year', 'Month Number', 'Month Name', 'Crime type']).agg(crime_count = ('crime_count', 'sum')).reset_index() 
)

In [128]:
# aggregating population and IMD separately

# this is where we have to de-deuplicate the data carefully
# I did it in its own step for easier bugfixing 
lsoa_level_lookup = lsoa_grain_df[['Falls within', 'LSOA code', 'Population', 'IMD_Decile']].drop_duplicates(subset = ['Falls within', 'LSOA code'])

# now we aggregate  population and IMD indices separately
force_population = lsoa_level_lookup.groupby('Falls within')['Population'].sum()
force_IMD = lsoa_level_lookup.groupby('Falls within')['IMD_Decile'].mean() # taking the mean here not the sum! 


In [129]:
# merging these with the main dataset

force_grain_df = force_grain_df.merge(
    force_population, on= 'Falls within', how = 'left')
force_grain_df = force_grain_df.merge(
    force_IMD, on= 'Falls within', how = 'left')

# renaming columns for accuracy
force_grain_df = force_grain_df.rename(columns = {'IMD_Decile' : 'Mean IMD Decile'})


In [130]:
# recomputing custom metrics using same method as above

force_grain_df['crime_rate_per_1000'] = (
    force_grain_df['crime_count'] / force_grain_df['Population']
) * 1000

month_totals_by_force = force_grain_df.groupby(
    ['Falls within', 'Year', 'Month Number'])['crime_count'].transform('sum')

force_grain_df['crime_category_share'] = (force_grain_df['crime_count'] / month_totals_by_force)


Now Im gonna perform some sanity checks to make sure that my aggregation worked as intended. I was running short on time due to some roadwork issues at this point, so this code is **AI generated**. I have used many of these same checks earlier in the project though, so its not something that I couldnt write myself.

In [131]:
pop_variation_check = force_grain_df.groupby('Falls within')['Population'].nunique()
print(pop_variation_check)

Falls within
Metropolitan Police Service    1
Northumbria Police             1
Surrey Police                  1
West Midlands Police           1
Name: Population, dtype: int64


Awesome, we only have a single population value per police force row.

In [132]:
imd_variation_check = force_grain_df.groupby('Falls within')['Mean IMD Decile'].nunique()
print(imd_variation_check)

Falls within
Metropolitan Police Service    1
Northumbria Police             1
Surrey Police                  1
West Midlands Police           1
Name: Mean IMD Decile, dtype: int64


Same thing for our mean IMD value. Now we're checking the force level population against a manual sum to make sure that its the right value.

In [133]:
manual_check = (
    lsoa_grain_df[['Falls within', 'LSOA code', 'Population']]
    .drop_duplicates(subset=['Falls within', 'LSOA code'])
    .groupby('Falls within')['Population']
    .sum()
)

comparison = force_grain_df.groupby('Falls within')['Population'].first().compare(manual_check)
print('Differences found:' if not comparison.empty else 'Match confirmed')
print(comparison)

Match confirmed
Empty DataFrame
Columns: [self, other]
Index: []


This confirms that our population values match up. The next checks are whether our crime percentage column sums to 1 per force, and whether crime count totals stay consistent between grains.

In [134]:
share_check = force_grain_df.groupby(['Falls within', 'Year', 'Month Number'])['crime_category_share'].sum()
print(share_check.describe())  # mean should be very close to 1.0

count    144.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
Name: crime_category_share, dtype: float64


Exactly as required, now the final check.

In [135]:
total_force_grain = force_grain_df['crime_count'].sum()
total_lsoa_grain = lsoa_grain_df['crime_count'].sum()

print(f'Total crimes (force grain): {total_force_grain:,}')
print(f'Total crimes (LSOA grain): {total_lsoa_grain:,}')
print(f'Match: {total_force_grain == total_lsoa_grain}')

Total crimes (force grain): 5,168,253
Total crimes (LSOA grain): 5,168,253
Match: True


All of these checks confirmed that the data aggregated properly. Now I can remove all the valus which have an out of bounds LSOA code from my LSOA grain, and then export the data. 

In [136]:
print(f'Rows before filtering: {len(lsoa_grain_df)}')

lsoa_grain_df = lsoa_grain_df[lsoa_grain_df['LSOA code'] != 'Out of bounds'].copy()

print(f'Rows after removing out of bounds: {len(lsoa_grain_df)}')

Rows before filtering: 1718865
Rows after removing out of bounds: 1701987


In [137]:
lsoa_grain_df.head()

,Falls within,LSOA code,Crime type,crime_count,Year,Month Number,Month Name,Population,IMD_Decile,crime_rate_per_1000,crime_category_share
12,Metropolitan Police Service,E01000006,Anti-social behaviour,2,2023,6,June,1930.0,4.0,1.036269,0.222222
13,Metropolitan Police Service,E01000006,Drugs,2,2023,6,June,1930.0,4.0,1.036269,0.222222
14,Metropolitan Police Service,E01000006,Vehicle crime,2,2023,6,June,1930.0,4.0,1.036269,0.222222
15,Metropolitan Police Service,E01000006,Violence and sexual offences,3,2023,6,June,1930.0,4.0,1.554404,0.333333
16,Metropolitan Police Service,E01000007,Anti-social behaviour,10,2023,6,June,3080.0,2.0,3.246753,0.175439


## Exporting Layer

In [138]:
force_grain_df.to_csv('Finished CSVs/force_grain.csv', index=False)
lsoa_grain_df.to_csv('Finished CSVs/lsoa_grain.csv', index=False)